# Project 1 - 用 DQN 玩 LunarLander-v3（改進實驗：評估 50 局 + 訓練 1000 局）

> 這份和 `Project1_LunarLander_DQN.ipynb` 幾乎相同，只改了兩個地方：
> 1. 第 2 格新增 `eval_games = 50`：訓練中每次評估從 10 局改成 50 局，挑最佳模型時比較不會被運氣騙。
> 2. 第 8 格訓練局數從 500 改成 1000：500 局時曲線還在上升，讓模型多學一點。
>
> 輸出檔名都加上 `_improved`，不會蓋掉原本 DQN 的結果。
### 改寫自 Deep Reinforcement Learning in Action 第 3 章（Listing 3.5 + 3.7）

```
本程式是由課本 Deep Reinforcement Learning in Action 第 3 章的程式
（Listing 3.5 Experience Replay + Listing 3.7 Target Network）改寫而來。
整體架構幾乎不變，主要差別只在「環境」與幾個超參數：

    Ch3 Gridworld                     →  本作業 LunarLander-v3
    ------------------------------------------------------------------
    狀態：4x4x4 one-hot 平面 = 64 維   →  8 維連續向量（位置、速度、角度、腳是否著地）
    動作：4 個（上下左右）              →  4 個（不動、左側引擎、主引擎、右側引擎）
    獎勵：每步 -1、目標 +10、陷阱 -10   →  官方連續獎勵：成功著陸約 +100~+140、墜毀 -100
    一局最多 50 步                      →  一局最多 1000 步（環境自己會 truncate）
    固定 epsilon = 0.3                  →  epsilon 從 1.0 指數衰減到 0.01
    記憶體 1000 筆、batch 200            →  記憶體 100,000 筆、batch 64
    gamma = 0.9                         →  gamma = 0.99（著陸需要看得更遠）
    MSELoss                             →  SmoothL1Loss（Huber loss，對大誤差較穩定）
```

**第一次使用前**，在 VS Code 終端機執行：
```
pip install torch gymnasium Box2D pygame matplotlib numpy
```
接著由上往下逐格執行。如果只想看訓練好的模型降落，執行完前面定義函數的儲存格後，直接跳到「載入已訓練好的模型」那一格。

In [ ]:
import copy
import random
from collections import deque

import gymnasium as gym
import matplotlib.pyplot as plt
import numpy as np
import torch

##### 1. 超參數

（對照 Ch3 Listing 3.2 / 3.5 / 3.7 的變數名稱，方便和課本比對）

In [ ]:
l1 = 8      # 輸入層：LunarLander 的狀態是 8 維（Ch3 是 64 維）
l2 = 128    # 第一個隱藏層
l3 = 128    # 第二個隱藏層
l4 = 4      # 輸出層：4 個動作各一個 Q 值

learning_rate = 5e-4   # Adam 學習率；比 Ch3 的 1e-3 小一點，避免 Q 值震盪
gamma = 0.99           # 折扣因子。著陸要考慮幾百步之後的結果，所以比 Ch3 的 0.9 大

epsilon_start = 1.0    # 一開始完全隨機探索
epsilon_end = 0.01     # 最低保留 1% 的隨機動作
epsilon_decay = 0.995  # 每局結束 epsilon *= 0.995，約 900 局後降到最低值

mem_size = 100_000     # Experience replay 容量（Ch3 是 1000）。LunarLander 一局就可能上百步，需要大很多
batch_size = 64        # 每次從記憶體隨機抽 64 筆來訓練
learn_start = 1_000    # 記憶體至少有 1000 筆才開始訓練，避免一開始樣本太少、太相關
sync_freq = 1_000      # 每 1000 個動作步，把主網路權重複製到 target network（Ch3 是 500）
train_every = 1        # 每走幾步訓練一次（1 = 每步都訓練，和 Ch3 相同）
grad_clip = 10.0       # 梯度裁剪上限，避免某一批 TD 誤差太大把網路權重炸掉
eval_every = 25        # 每 25 局用貪婪策略評估一次，保留最好的模型
eval_games = 50        # 改進：每次評估測 50 局（原本 10 局）。局數越多，平均分數的雜訊越小，挑到的最佳模型越可靠

action_names = {0: "不動", 1: "左側引擎", 2: "主引擎", 3: "右側引擎"}  # 對應 Ch3 的 action_set

##### 2. Q-network（對應 Listing 3.2）

輸入一個狀態 s，一次輸出 4 個動作的 Q(s, a)。 最後一層不加激活函數，因為 Q 值可正可負。

In [ ]:
def build_model():
    return torch.nn.Sequential(
        torch.nn.Linear(l1, l2),
        torch.nn.ReLU(),
        torch.nn.Linear(l2, l3),
        torch.nn.ReLU(),
        torch.nn.Linear(l3, l4),
    )

##### 3. Epsilon-greedy 選動作（對應 Listing 3.3 的 # I）

機率 epsilon 隨機探索，否則選 Q 值最大的動作。

In [ ]:
def select_action(model, state, epsilon):
    if random.random() < epsilon:
        return np.random.randint(0, l4)  # 探索
    with torch.no_grad():  # 選動作只需要前向傳播，不需要記錄梯度
        qval = model(state)
    return int(torch.argmax(qval).item())  # 利用

##### 4. 從 replay 抽一批資料，更新一次主網路

（對應 Listing 3.5 的 # J ~ # N，以及 Listing 3.7 改用 model2 算目標）

In [ ]:
def train_step(model, model2, replay, optimizer, loss_fn):
    # J：隨機抽 batch_size 筆經驗，打破連續步驟之間的相關性
    minibatch = random.sample(replay, batch_size)

    # K：把每筆 (s, a, r, s', done) 拆開，各自疊成一個 batch tensor
    state1_batch = torch.cat([s1 for (s1, a, r, s2, d) in minibatch])           # (64, 8)
    action_batch = torch.tensor([a for (s1, a, r, s2, d) in minibatch])         # (64,)
    reward_batch = torch.tensor([r for (s1, a, r, s2, d) in minibatch], dtype=torch.float32)
    state2_batch = torch.cat([s2 for (s1, a, r, s2, d) in minibatch])           # (64, 8)
    done_batch = torch.tensor([d for (s1, a, r, s2, d) in minibatch], dtype=torch.float32)

    # L：用主網路重新計算「目前」的 Q(s, ·)，這一項要保留梯度
    Q1 = model(state1_batch)                                                    # (64, 4)

    # M：用 target network 計算 Q(s', ·)，不需要梯度（Listing 3.7 # B）
    with torch.no_grad():
        Q2 = model2(state2_batch)                                               # (64, 4)

    # N：Q-learning 目標值  Y = r + gamma * (1 - done) * max_a' Q_target(s', a')
    #    done = 1（真的墜毀或著陸）時沒有未來，Y 只剩 r
    Y = reward_batch + gamma * (1 - done_batch) * torch.max(Q2, dim=1)[0]       # (64,)

    # 從 Q1 的 4 個輸出中，挑出「當時實際做的那個動作」的 Q 值
    X = Q1.gather(dim=1, index=action_batch.long().unsqueeze(dim=1)).squeeze()  # (64,)

    loss = loss_fn(X, Y.detach())  # Y 當作常數，不讓梯度流進目標值

    optimizer.zero_grad()
    loss.backward()
    # 新增：梯度裁剪。LunarLander 墜毀時一次 -100，TD 誤差可能很大，裁剪可以讓訓練更穩
    torch.nn.utils.clip_grad_norm_(model.parameters(), grad_clip)
    optimizer.step()
    return loss.item()

##### 5. 主訓練迴圈（對應 Listing 3.7 整段）

In [ ]:
def train(episodes, seed):
    # 固定所有隨機種子，讓結果可以重現（報告時比較不同設定比較公平）
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)

    env = gym.make("LunarLander-v3")

    model = build_model()                          # 主網路：負責選動作、被梯度更新
    model2 = copy.deepcopy(model)                  # Listing 3.7 # A：target network
    model2.load_state_dict(model.state_dict())     # Listing 3.7 # B：權重一開始相同
    loss_fn = torch.nn.SmoothL1Loss()              # Huber loss：誤差小時像 MSE，誤差大時像 L1
    optimizer = torch.optim.Adam(model.parameters(), lr=learning_rate)

    replay = deque(maxlen=mem_size)                # Listing 3.5 # C：滿了會自動丟掉最舊的經驗
    epsilon = epsilon_start
    losses = []            # 每次訓練的 loss（對應 Ch3 的 losses）
    episode_rewards = []   # 每局總獎勵，用來判斷有沒有學會著陸
    j = 0                  # 全域步數計數器，用來決定何時同步 target network
    best_score = -float("inf")                     # 目前貪婪策略評估的最佳平均分數
    best_state = copy.deepcopy(model.state_dict()) # 最佳分數對應的模型權重

    for i in range(episodes):
        # gymnasium 的 reset 會回傳 (observation, info)
        # Ch3 是 game.board.render_np().reshape(1, 64)；這裡狀態本來就是 8 維向量，
        # 只需要 reshape 成 (1, 8) 當作 batch size = 1 的輸入
        state1_, _ = env.reset(seed=seed + i)
        state1 = torch.from_numpy(state1_).float().unsqueeze(0)
        total_reward = 0.0
        done = False

        while not done:
            j += 1
            action_ = select_action(model, state1, epsilon)

            # gymnasium 的 step 回傳 5 個值：
            #   terminated：真的結束（墜毀、著陸後靜止、飛出畫面）→ 沒有未來獎勵
            #   truncated ：只是步數到上限（1000 步）被強制停止 → 其實還有未來
            # 這對應 Ch3 註解提到的「max_moves 不是 absorbing state」
            state2_, reward, terminated, truncated, _ = env.step(action_)
            state2 = torch.from_numpy(state2_).float().unsqueeze(0)
            done = terminated or truncated

            # Listing 3.5 # G、# H：存一筆經驗。注意存的是 terminated 而不是 done，
            # 這樣被 truncate 的最後一步，目標值仍會加上 gamma * max Q(s')
            replay.append((state1, action_, reward, state2, terminated))
            state1 = state2
            total_reward += reward

            # Listing 3.5 # I：記憶體夠多才開始批次訓練
            if len(replay) >= learn_start and j % train_every == 0:
                losses.append(train_step(model, model2, replay, optimizer, loss_fn))

            # Listing 3.7 # C：每 sync_freq 步同步一次 target network
            if j % sync_freq == 0:
                model2.load_state_dict(model.state_dict())

        # 每局結束後衰減 epsilon（Ch3 Listing 3.3 # R 是線性遞減，這裡改成指數遞減）
        epsilon = max(epsilon_end, epsilon * epsilon_decay)
        episode_rewards.append(total_reward)

        # 新增：每 eval_every 局用「完全不探索」的策略測 eval_games 局，保留表現最好的那一版權重。
        # 原因：訓練時的分數有 epsilon 隨機動作幫忙，貪婪策略（epsilon=0）有時會學成
        # 「一直懸停不降落」直到 1000 步上限，所以要另外用貪婪策略評估，挑最好的模型。
        if (i + 1) % eval_every == 0:
            eval_score = test_model(model, games=eval_games, seed=50_000, verbose=False).mean()  # 改進：10 局 → eval_games（50）局
            if eval_score > best_score:
                best_score = eval_score
                best_state = copy.deepcopy(model.state_dict())
                print(f"  -> 第 {i + 1} 局貪婪策略平均 {eval_score:.2f}，目前最佳，已保存")

        if (i + 1) % 20 == 0:
            avg_r = np.mean(episode_rewards[-100:])
            avg_l = np.mean(losses[-1000:]) if losses else float("nan")
            print(
                f"Episode {i + 1:4d} | 近100局平均獎勵 {avg_r:8.2f} | "
                f"近1000次平均 loss {avg_l:.4f} | epsilon {epsilon:.3f}"
            )

    env.close()
    model.load_state_dict(best_state)  # 訓練結束後換回評估分數最高的那一版
    print(f"使用貪婪策略評估最佳的模型（評估平均 {best_score:.2f}）")
    return model, np.array(losses), np.array(episode_rewards)

##### 6. 測試（對應 Listing 3.4 test_model 與 3.6 計算勝率）

測試時 epsilon = 0，完全依照網路選最好的動作。 LunarLander 官方標準：一局總獎勵 >= 200 視為「成功解決」。

In [ ]:
def test_model(model, games=100, render=False, seed=10_000, verbose=True):
    env = gym.make("LunarLander-v3", render_mode="human" if render else None)
    scores = []
    for g in range(games):
        state_, _ = env.reset(seed=seed + g)
        state = torch.from_numpy(state_).float().unsqueeze(0)
        done, total = False, 0.0
        while not done:
            action_ = select_action(model, state, epsilon=0.0)
            state_, reward, terminated, truncated, _ = env.step(action_)
            state = torch.from_numpy(state_).float().unsqueeze(0)
            total += reward
            done = terminated or truncated
        scores.append(total)
    env.close()
    scores = np.array(scores)
    if verbose:
        print(f"測試 {games} 局：平均分數 {scores.mean():.2f}，"
              f"成功率（>=200 分）{(scores >= 200).mean() * 100:.1f}%")
    return scores


# 對應 Ch3 的 running_mean：把雜訊很大的曲線做移動平均，比較看得出趨勢
def running_mean(x, N=50):
    if len(x) < N:
        return x
    return np.convolve(x, np.ones(N) / N, mode="valid")


def save_plots(losses, rewards):
    fig, ax = plt.subplots(1, 2, figsize=(14, 5))
    ax[0].plot(running_mean(losses, 500))
    ax[0].set_xlabel("Training steps")
    ax[0].set_ylabel("Loss (running mean, N=500)")
    ax[0].set_title("Loss")
    ax[1].plot(rewards, alpha=0.3, label="each episode")
    ax[1].plot(range(99, len(rewards)), running_mean(rewards, 100), label="mean of last 100")
    ax[1].axhline(200, color="r", linestyle="--", label="solved (200)")
    ax[1].set_xlabel("Episode")
    ax[1].set_ylabel("Total reward")
    ax[1].set_title("Episode reward")
    ax[1].legend()
    fig.tight_layout()
    fig.savefig("training_curves_improved.png", dpi=120)

##### 7. 開始訓練（約 10~20 分鐘）

會每 20 局印一次進度；結束後自動換成貪婪評估最好的那一版模型。

In [ ]:
model, losses, rewards = train(episodes=1000, seed=0)  # 改進：500 局 → 1000 局
torch.save(model.state_dict(), "dqn_lunarlander_improved.pt")  # 存檔，之後可以直接載入
print(f"最後 500 次訓練的平均 loss：{losses[-500:].mean():.4f}")

##### 8. 畫出 loss 與每局獎勵曲線（對應 Ch3 的 running_mean 圖）

In [ ]:
save_plots(losses, rewards)  # 同時存成 training_curves_improved.png
plt.show()

##### 9. 載入已訓練好的模型

如果不想重新訓練，可以跳過第 7、8 格，直接執行這一格，載入資料夾裡的 `dqn_lunarlander.pt`。

In [ ]:
model = build_model()
model.load_state_dict(torch.load("dqn_lunarlander_improved.pt"))

##### 10. 測試 100 局（對應 Listing 3.6 計算勝率）

In [ ]:
scores = test_model(model, games=100)

In [ ]:
plt.figure(figsize=(10, 4))
plt.bar(range(1, 101), scores, color=["tab:blue" if s >= 200 else "tab:red" for s in scores])
plt.axhline(200, color="r", linestyle="--", label="solved (200)")
plt.axhline(scores.mean(), color="tab:orange", label=f"mean = {scores.mean():.1f}")
plt.xlabel("Test game")
plt.ylabel("Total reward")
plt.title("DQN (eval 50 + 1000 episodes): 100 test games")
plt.legend()
plt.tight_layout()
plt.savefig("test_scores_improved.png", dpi=120)
plt.show()

##### 11. 開視窗看飛船降落（對應 Listing 3.4 test_model 的 display）

會跳出一個 pygame 視窗播放 3 局，播完自動關閉。

In [ ]:
test_model(model, games=3, render=True)

In [ ]:
seed_scores = {sd: test_model(model, games=1, seed=sd, verbose=False)[0] for sd in range(10000, 10040)}
best = max(seed_scores, key=seed_scores.get)
print(f"最好的 seed：{best}，分數 {seed_scores[best]:.2f}")
top3 = sorted(seed_scores, key=seed_scores.get, reverse=True)[:3]
for rank, sd in enumerate(top3, 1):
    print(f"第 {rank} 名：seed {sd}，分數 {seed_scores[sd]:.2f}")

In [ ]:
# 12. 把一局降落過程錄成 GIF 動畫，直接顯示在筆記本裡（也可以放進簡報）
from PIL import Image
from IPython.display import Image as ShowImage, display

env = gym.make("LunarLander-v3", render_mode="rgb_array")  # rgb_array：每一步回傳畫面圖片，不開視窗
state_, _ = env.reset(seed=best)                            # 用上一格找到的最好 seed
frames, total, done = [env.render()], 0.0, False
while not done:
    state = torch.from_numpy(state_).float().unsqueeze(0)
    action_ = select_action(model, state, epsilon=0.0)       # 測試時完全不探索
    state_, reward, terminated, truncated, _ = env.step(action_)
    total += reward
    done = terminated or truncated
    frames.append(env.render())                              # 收集每一格畫面
env.close()

imgs = [Image.fromarray(f) for f in frames[::2]]             # 每 2 格取 1 格，檔案比較小
imgs[0].save("lunarlander_improved.gif", save_all=True, append_images=imgs[1:], duration=40, loop=0)
print(f"seed {best} 這一局分數 {total:.2f}，共 {len(frames)} 步，已存成 lunarlander_improved.gif")
display(ShowImage(filename="lunarlander_improved.gif"))